# HIT401 Simple Scam Email Prototype

This notebook creates a small research demonstration using the same modern approach as Part 4:

**Email → MiniLM embedding → Logistic Regression → Scam or legitimate**

It also uses separate keyword rules to show behavioural cues. The cue rules help describe the email, but they do not explain exactly why the model made its prediction.

Run each cell from top to bottom. Use a **T4 GPU** if Colab offers one.

In [ ]:
# Install the required libraries.
!pip -q install "sentence-transformers>=3.0,<6" "gradio>=4.44,<6"

In [ ]:
# Import the tools used by the prototype.
import os
import re
import html
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import gradio as gr
from google.colab import files
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Libraries loaded successfully.")

## Step 1 — Upload the prepared data

Upload the file created in Part 1:

`hit401_prepared_data.zip`

It must contain `training_data.csv` and `external_test_data.csv`.

In [ ]:
uploaded = files.upload()

zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if not zip_names:
    raise ValueError("Please upload hit401_prepared_data.zip")

zip_name = zip_names[0]
extract_dir = Path("prototype_data")
extract_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(extract_dir)

csv_files = list(extract_dir.rglob("*.csv"))
print("CSV files found:", [p.name for p in csv_files])

In [ ]:
# Find and validate the two prepared CSV files.
def find_file(filename):
    matches = list(extract_dir.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"{filename} was not found inside the ZIP")
    return matches[0]

training = pd.read_csv(find_file("training_data.csv"))
external = pd.read_csv(find_file("external_test_data.csv"))

def prepare_frame(frame, name):
    frame = frame.copy()
    if "text" not in frame.columns or "label" not in frame.columns:
        raise ValueError(f"{name} must contain text and label columns. Found: {list(frame.columns)}")
    frame["text"] = frame["text"].fillna("").astype(str).str.strip()
    frame["label"] = pd.to_numeric(frame["label"], errors="raise").astype(int)
    frame = frame.loc[frame["text"].ne("") & frame["label"].isin([0, 1])].copy()
    return frame

training = prepare_frame(training, "training_data.csv")
external = prepare_frame(external, "external_test_data.csv")

print(f"Training emails: {len(training):,}")
print(training["label"].map({0: "Legitimate", 1: "Scam"}).value_counts())
print(f"\nExternal-test emails: {len(external):,}")
print(external["label"].map({0: "Legitimate", 1: "Scam"}).value_counts())

## Step 2 — Load MiniLM and train the classifier

MiniLM is downloaded from Hugging Face. It converts every email into 384 contextual numbers. We keep MiniLM frozen and train only Logistic Regression using the labelled training emails.

This is the slowest section. It may take several minutes.

In [ ]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

minilm = SentenceTransformer(MODEL_NAME)
# Limiting the length keeps the demonstration manageable. Longer emails are truncated.
minilm.max_seq_length = 256

print("Creating MiniLM training embeddings...")
X_train = minilm.encode(
    training["text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

classifier = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42,
)
classifier.fit(X_train, training["label"].to_numpy())

print("Prototype model trained.")
print("Embedding shape:", X_train.shape)

In [ ]:
# Optional verification using the prepared external test data.
print("Creating external-test embeddings...")
X_external = minilm.encode(
    external["text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

external_predictions = classifier.predict(X_external)
y_external = external["label"].to_numpy()

print(f"External accuracy:  {accuracy_score(y_external, external_predictions):.2%}")
print(f"Scam precision:     {precision_score(y_external, external_predictions, zero_division=0):.2%}")
print(f"Scam recall:        {recall_score(y_external, external_predictions, zero_division=0):.2%}")
print(f"Scam F1-score:      {f1_score(y_external, external_predictions, zero_division=0):.2%}")
print("Small differences from an earlier run can occur if software versions differ.")

## Step 3 — Add simple behavioural cue rules

These rules look for selected words and phrases. They are separate from the MiniLM prediction.

In [ ]:
CUE_TERMS = {
    "Urgency": ["urgent", "immediately", "act now", "as soon as possible", "limited time", "today"],
    "Authority": ["official", "government", "police", "manager", "administrator", "department"],
    "Fear or threat": ["suspended", "blocked", "penalty", "warning", "legal action", "terminate"],
    "Reward or gain": ["winner", "prize", "reward", "lottery", "bonus", "inheritance"],
    "Trust or credibility": ["trusted", "verified", "secure", "guaranteed", "confidential", "legitimate"],
    "Financial request": ["payment", "bank", "transfer", "money", "fee", "credit card", "bitcoin"],
    "Credential request": ["password", "login", "username", "verify your account", "security code", "pin"],
    "Relationship or emotion": ["dear friend", "beloved", "help me", "family", "love", "kindness"],
}

def detect_cues(text):
    lower = text.lower()
    found = {}
    for cue, terms in CUE_TERMS.items():
        matches = []
        for term in terms:
            if re.search(r"\b" + re.escape(term) + r"\b", lower, flags=re.IGNORECASE):
                matches.append(term)
        if matches:
            found[cue] = matches
    return found

ALL_TERMS = sorted({term for terms in CUE_TERMS.values() for term in terms}, key=len, reverse=True)
HIGHLIGHT_PATTERN = re.compile(
    r"\b(" + "|".join(re.escape(term) for term in ALL_TERMS) + r")\b",
    flags=re.IGNORECASE,
)

def highlight_terms(text):
    safe_text = html.escape(text)
    return HIGHLIGHT_PATTERN.sub(
        lambda match: (
            '<mark style="background:#ffe08a;padding:2px 4px;border-radius:3px;">'
            + match.group(0)
            + "</mark>"
        ),
        safe_text,
    ).replace("\n", "<br>")

In [ ]:
# Test the complete pipeline before opening the interface.
def analyze_email(email_text):
    email_text = str(email_text or "").strip()
    if len(email_text) < 10:
        return (
            "Please enter a longer email",
            "No probability available",
            "No cues checked",
            "<p>Please paste an email containing at least 10 characters.</p>",
        )

    embedding = minilm.encode(
        [email_text],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    scam_probability = float(classifier.predict_proba(embedding)[0, 1])
    prediction = "SCAM" if scam_probability >= 0.50 else "LEGITIMATE"

    cues = detect_cues(email_text)
    if cues:
        cue_lines = [f"{cue}: {', '.join(terms)}" for cue, terms in cues.items()]
        cue_text = "\n".join(cue_lines)
    else:
        cue_text = "No selected behavioural cue words were found."

    result = f"Prediction: {prediction}"
    probability = f"Estimated scam probability: {scam_probability:.1%}"
    highlighted = (
        "<div style='font-family:Arial;line-height:1.6'>"
        "<h3>Highlighted email</h3>"
        f"<p>{highlight_terms(email_text)}</p>"
        "<hr><p><b>Research demonstration only.</b> "
        "Do not use this result as the only basis for a security decision.</p></div>"
    )
    return result, probability, cue_text, highlighted

example = "Urgent: verify your bank account immediately or it will be suspended."
print(analyze_email(example)[:3])

## Step 4 — Launch the prototype

After running the next cell, open the public Gradio link. The link normally remains active only while this Colab runtime is running.

In [ ]:
demo = gr.Interface(
    fn=analyze_email,
    inputs=gr.Textbox(
        label="Paste an email",
        lines=10,
        placeholder="Paste the email subject and message here...",
    ),
    outputs=[
        gr.Textbox(label="Model prediction"),
        gr.Textbox(label="Probability"),
        gr.Textbox(label="Behavioural cues", lines=8),
        gr.HTML(label="Highlighted phrases"),
    ],
    title="Scam Email Explorer",
    description=(
        "MiniLM creates contextual features, Logistic Regression predicts the class, "
        "and separate keyword rules display selected behavioural cues."
    ),
    examples=[
        ["Urgent: verify your bank account immediately or it will be suspended."],
        ["Hi team, the meeting has moved to 2 pm tomorrow. Please see the updated agenda."],
    ],
    flagging_mode="never",
)

demo.launch(share=True, debug=False)

## What to say during the demonstration

> “The user pastes an email. MiniLM converts the email into 384 contextual features, and our trained Logistic Regression classifier estimates whether it is a scam. Separate keyword rules display cues such as urgency, authority and financial requests. The prototype is a research demonstration, not a finished security product.”

### Important limitation

The behavioural highlights are rule-based. They do not show exactly which words caused the MiniLM classifier to make its decision.